# LibAR 대림 테스트 (7장 멀티프레임, GPU)

**파이프라인:** 색 기반 라벨 검출(파란띠, 학습 불필요) → PaddleOCR(GPU) → 일반 청구기호 매칭(분류+저자기호 퍼지) → 행별 LIS 오배열 → 멀티프레임 집계

**사용법 (순서 중요):**
1. GPU(T4) 런타임 선택
2. **셀1(설치) 실행 → [런타임 → 세션 다시 시작]** ← 필수! (안 하면 PDX 초기화 에러)
3. 셀2(업로드): `daelim_colab_package.zip` 업로드 — 이미 업로드했다면 unzip 줄만 다시 실행돼도 OK
4. 셀3(정의) → 셀4(7장 실행) → 셀5(집계) → 결과 zip 자동 다운로드

※ 세션 재시작 시 설치와 파일은 유지되고 메모리(변수)만 초기화됩니다.

In [ ]:
# GPU 설치 — torch를 먼저 제거해 NCCL 충돌 원천 차단 (우리 파이프라인은 torch 불필요)
# ⚠️ 이 셀 실행 후 [런타임 → 세션 다시 시작] 하고 셀2부터 진행!
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr opencv-python-headless pillow
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2(업로드)부터 실행하세요.')
print('   (세션 재시작은 파일 유지 / "런타임 삭제"만 파일이 지워집니다)')

In [ ]:
from google.colab import files
up = files.upload()   # daelim_colab_package.zip 업로드
!unzip -oq daelim_colab_package.zip && ls daelim | head

In [ ]:
# ── 파이프라인 정의 (검출/매칭/LIS/제목) ──
import cv2, numpy as np, re, csv, json, difflib, unicodedata, glob, os, time
from PIL import Image, ImageDraw

def nn(s): return re.sub(r'[^0-9A-Za-z가-힣.]','',unicodedata.normalize('NFC',str(s)))
def ntitle(s): return re.sub(r'[^0-9A-Za-z가-힣]','',unicodedata.normalize('NFC',str(s))).lower()

def detect_labels(bgr):
    H,W=bgr.shape[:2]
    hsv=cv2.cvtColor(bgr,cv2.COLOR_BGR2HSV)
    blue=((hsv[:,:,0]>=90)&(hsv[:,:,0]<=135)&(hsv[:,:,1]>50)&(hsv[:,:,2]>60)).astype(np.uint8)
    white=((hsv[:,:,1]<75)&(hsv[:,:,2]>135)).astype(np.uint8)
    rowfrac=blue.mean(axis=1)
    thr=max(0.04,float(np.percentile(rowfrac,90))*0.5)
    on=rowfrac>thr; bands=[]; st=None
    for i,g in enumerate(list(on)+[False]):
        if g and st is None: st=i
        elif not g and st is not None:
            if 10<=i-st<=90: bands.append((st,i))
            st=None
    merged=[]
    for b in bands:
        if merged and b[0]-merged[-1][1]<12: merged[-1]=(merged[-1][0],b[1])
        else: merged.append(b)
    bands=[b for b in merged if 10<=b[1]-b[0]<=110]
    labels=[]
    for (by0,by1) in bands:
        colfrac=blue[by0:by1,:].mean(axis=0); onc=colfrac>0.45; segs=[]; st=None
        for i,g in enumerate(list(onc)+[False]):
            if g and st is None: st=i
            elif not g and st is not None:
                if i-st>=14: segs.append((st,i))
                st=None
        bh=by1-by0; uy0=max(0,by0-int(bh*2.9)); uy1=by0
        for (sx0,sx1) in segs:
            reg=white[uy0:uy1,sx0:sx1]
            if reg.size==0: continue
            wf=reg.mean(axis=0); onw=wf>0.45; st2=None; found=False
            for i,g in enumerate(list(onw)+[False]):
                if g and st2 is None: st2=i
                elif not g and st2 is not None:
                    if i-st2>=13: labels.append((sx0+st2,uy0,sx0+i,by1)); found=True
                    st2=None
            if not found: labels.append((sx0,uy0,sx1,by1))
    return labels,bands

cat=[]
for r in csv.DictReader(open('daelim/daelim_catalog.csv',encoding='utf-8-sig')):
    p=r['call_number'].strip().split('-')
    if len(p)<2: continue
    m=re.match(r'^[가-힣A-Z]*([\d.]+)$',p[0].strip())
    main=r['title'].split(':')[0].strip()
    cat.append({'call':r['call_number'].strip(),'cls':m.group(1) if m else p[0],
                'author':nn(p[1]),'title':r['title'],'t':ntitle(main),'status':r['status']})
by_cls={}
for c in cat: by_cls.setdefault(c['cls'],[]).append(c)
print('카탈로그',len(cat),'권')

def match(txt):
    t=nn(txt)
    mc=re.search(r'8[234]\d(\.\d+)?',t)
    cands=by_cls.get(mc.group(0),cat) if mc else cat
    clsv=mc.group(0) if mc else None
    ma=re.findall(r'[가-힣][\d]{1,3}[가-힣]?',t)
    best,bs=None,0.0
    for c in cands:
        s=0.0
        for a in ma: s=max(s,difflib.SequenceMatcher(None,a,c['author']).ratio())
        if clsv and s>0: s+=0.15
        if s>bs: best,bs=c,s
    return (best,bs) if bs>=0.62 else (None,bs)

def match_title(txt,thr=0.60):   # 0.55는 짧은 잡음('IIN' 등) 오탐 → 0.60
    t=ntitle(txt)
    if len(t)<4: return None,0.0
    best,bs=None,0.0
    for c in cat:
        if not c['t']: continue
        lm=difflib.SequenceMatcher(None,t,c['t']).find_longest_match(0,len(t),0,len(c['t']))
        p=lm.size/min(len(t),len(c['t']))
        s=0.6*p+0.4*difflib.SequenceMatcher(None,t,c['t']).ratio()
        if s>bs: best,bs=c,s
    return (best,bs) if bs>=thr else (None,bs)

def sortkey(call):
    p=call.split('-'); cls=re.sub(r'^[가-힣A-Z]+','',p[0]); vol=0
    if len(p)>2:
        mv=re.search(r'\d+',p[2]); vol=int(mv.group(0)) if mv else 0
    return (float(cls) if re.match(r'^[\d.]+$',cls) else 999, p[1] if len(p)>1 else '', vol)

def lis_mis(keys):
    n=len(keys)
    if n==0: return set()
    Ln=[1]*n; P=[-1]*n
    for i in range(n):
        for j in range(i):
            if keys[j]<=keys[i] and Ln[j]+1>Ln[i]: Ln[i]=Ln[j]+1; P[i]=j
    e=max(range(n),key=lambda i:Ln[i]); keep=set()
    while e!=-1: keep.add(e); e=P[e]
    return set(range(n))-keep
print('준비 완료')

In [ ]:
# ── 7장 전체 실행: 이중 인식 (스트립-청크 청구기호 OCR + 제목 복구 패스) ──
from paddleocr import PaddleOCR
ocr = PaddleOCR(lang='korean', use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False)
os.makedirs('out', exist_ok=True)
CHUNK, OV, UP = 1250, 120, 3

def ocr_strip_tokens(bgr, bands):
    H, W = bgr.shape[:2]
    toks = []
    for (by0, by1) in bands:
        bh = by1 - by0
        uy0 = max(0, by0 - int(bh*3.2)); uy1 = min(H, by1 + 4)
        for cx0 in range(0, W, CHUNK - OV):
            chunk = bgr[uy0:uy1, cx0:min(W, cx0+CHUNK)]
            if chunk.size == 0 or chunk.shape[1] < 40: continue
            up = cv2.resize(chunk, None, fx=UP, fy=UP, interpolation=cv2.INTER_LANCZOS4)
            res = ocr.predict(up)
            if not res: continue
            r = res[0]
            for t, p in zip(r.get('rec_texts', []), r.get('rec_polys', [])):
                xs = [q[0] for q in p]; ys = [q[1] for q in p]
                toks.append((cx0 + (sum(xs)/len(xs))/UP, uy0 + (sum(ys)/len(ys))/UP, t))
    seen, out = set(), []
    for x, y, t in toks:
        k = (round(x/12), round(y/12), t)
        if k in seen: continue
        seen.add(k); out.append((x, y, t))
    return out

all_results = {}
for src in sorted(glob.glob('daelim/*.jpg')):
    name = os.path.basename(src).split('.')[0][-2:]
    im = Image.open(src).convert('RGB')
    bgr = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2BGR)
    labels, bands = detect_labels(bgr)
    t0 = time.time()
    toks = ocr_strip_tokens(bgr, bands)
    rows = []
    for b in labels:
        x0, y0, x1, y1 = b
        mine = [(y, t) for (x, y, t) in toks if x0-3 <= x <= x1+3 and y0-6 <= y <= y1+6]
        mine.sort()
        txt = ' '.join(t for _, t in mine)
        yc = (y0+y1)/2
        band = next((i for i, (a, c) in enumerate(bands) if a-200 <= yc <= c+40), -1)
        row, sc = match(txt)
        rows.append({'box': list(b), 'band': band, 'read': txt,
                     'call': row['call'] if row else None, 'score': round(sc, 2), 'how': '청구기호' if row else None})
    # ── 제목 복구 패스: 라벨 위 세로 기둥 90°회전 OCR ──
    tops = sorted({r['box'][1] for r in rows})
    gaps = [b-a for a, b in zip(tops, tops[1:]) if b-a > 300]
    ROWH = int(np.median(gaps)) if gaps else 650
    n_rec = n_dual = 0
    for r in rows:
        x0, y0, x1, y1 = r['box']
        ty0 = max(0, y0-int(ROWH*0.78)); ty1 = y0-8
        tc = bgr[ty0:ty1, max(0, x0-5):x1+5]
        if tc.size == 0: continue
        tc = cv2.rotate(tc, cv2.ROTATE_90_COUNTERCLOCKWISE)
        tc = cv2.resize(tc, None, fx=2, fy=2, interpolation=cv2.INTER_LANCZOS4)
        res = ocr.predict(tc)
        ttxt = ' '.join(res[0].get('rec_texts', [])) if res else ''
        trow, tsc = match_title(ttxt)
        if trow:
            if r['call'] and r['call'] == trow['call']: r['how'] = '이중확정'; n_dual += 1
            elif not r['call']:
                r['call'] = trow['call']; r['how'] = '제목복구'; r['score'] = round(tsc, 2); n_rec += 1
    matched = sum(1 for r in rows if r['call'])
    n_mis = 0
    for bi in range(len(bands)):
        seq = [r for r in rows if r['band'] == bi and r['call']]
        seq.sort(key=lambda r: r['box'][0])
        mis = lis_mis([sortkey(r['call']) for r in seq])
        for j, r in enumerate(seq):
            r['mis'] = j in mis
            if j in mis: n_mis += 1
    print(f'[{name}] 라벨 {len(labels)} · 매칭 {matched} ({matched/max(1,len(labels))*100:.0f}%) [제목복구 +{n_rec} · 이중확정 {n_dual}] · 오배열 {n_mis} · {time.time()-t0:.0f}s')
    d = ImageDraw.Draw(im)
    for r in rows:
        c = (150,150,150) if not r['call'] else ((235,60,60) if r.get('mis') else ((50,130,240) if r.get('how')=='제목복구' else (40,190,90)))
        d.rectangle(r['box'], outline=c, width=4)
    im.save(f'out/daelim_{name}_ar.jpg', quality=86)
    json.dump(rows, open(f'out/daelim_{name}.json','w',encoding='utf-8'), ensure_ascii=False)
    all_results[name] = rows

In [ ]:
# ── 멀티프레임 집계: 사진별 인식 합집합 + 투표 ──
from collections import Counter
votes = Counter()
for name, rows in all_results.items():
    for call in {r['call'] for r in rows if r['call']}:
        votes[call] += 1
u1 = sum(1 for v in votes.values() if v>=1)
u2 = sum(1 for v in votes.values() if v>=2)
u3 = sum(1 for v in votes.values() if v>=3)
print(f'인식된 고유 도서: 1표+ {u1}권 · 2표+ {u2}권(신뢰) · 3표+ {u3}권(높은신뢰)')
print(f'(카탈로그 456권 중, 사진에 보이는 구간만 해당)')
# 대출중 도서와 대조 (결번 해석 데모)
loaned = {c['call'] for c in cat if '대출' in c['status']}
seen = set(votes)
print(f'\n대출중(관외대출) {len(loaned)}권 중 서가에서 발견된 것: {len(loaned & seen)}권 (0이 정상)')
json.dump({'votes': dict(votes)}, open('out/aggregate.json','w',encoding='utf-8'), ensure_ascii=False)
!cd out && zip -q -r ../daelim_results.zip . && cd ..
from google.colab import files
files.download('daelim_results.zip')